#### In Silver we:

1. Remove duplicates
2. Handle updates
3. Process only NEW / CHANGED data (incremental)

In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "uat", "prod"])

env = dbutils.widgets.get("env")

catalog = env
bronze = "bronze"
silver = "silver"

In [0]:
orders = spark.table(f"{catalog}.{bronze}.bronze_orders")
customers = spark.table(f"{catalog}.{bronze}.bronze_customers")
products = spark.table(f"{catalog}.{bronze}.bronze_products")
order_items = spark.table(f"{catalog}.{bronze}.bronze_order_items")

In [0]:
from pyspark.sql.functions import col

# First run → full load
last_run_time = "1900-01-01"

orders_inc = orders.filter(
    (col("created_at") > last_run_time) |
    (col("updated_at") > last_run_time)
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("order_id").orderBy(col("updated_at").desc())

orders_dedup = orders_inc \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")

In [0]:
from pyspark.sql.functions import broadcast

silver_df = orders_dedup.alias("o") \
    .join(broadcast(customers.alias("c")), "customer_id") \
    .join(order_items.alias("oi"), "order_id") \
    .join(broadcast(products.alias("p")), "product_id")

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.select(
    "o.order_id",
    "o.customer_id",
    "c.name",
    "c.state",
    "oi.product_id",
    "p.product_name",
    "p.category",
    "oi.quantity",
    col("oi.price").alias("sale_price"),
    "o.order_date",
    "o.order_status",
    "o.created_at",
    "o.updated_at"
)

In [0]:
silver_table = f"{catalog}.{silver}.silver_orders_enriched"

if not spark.catalog.tableExists(silver_table):
    
    # First load
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(silver_table)
    
    print(f"✅ Created {silver_table}")

else:
    
    # Incremental load (UPSERT)
    silver_df.createOrReplaceTempView("updates")
    
    spark.sql(f"""
    MERGE INTO {silver_table} t
    USING updates s
    ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    
    print(f"✅ Incremental load applied to {silver_table}")

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def process_silver_basic(table_name, primary_key):
    
    env = dbutils.widgets.get("env")
    
    bronze_table = f"{env}.bronze.bronze_{table_name}"
    silver_table = f"{env}.silver.silver_{table_name}"
    
    df = spark.table(bronze_table)
    
    # INCREMENTAL LOGIC
    last_run_time = "1900-01-01"   # later from control table
    
    df_inc = df.filter(
        (col("created_at") > last_run_time) |
        (col("updated_at") > last_run_time)
    )
    
    # DEDUPLICATION
    window = Window.partitionBy(primary_key).orderBy(col("updated_at").desc())
    
    df_dedup = df_inc \
        .withColumn("rn", row_number().over(window)) \
        .filter("rn = 1") \
        .drop("rn")
    
    # MERGE (UPSERT)
    if not spark.catalog.tableExists(silver_table):
        
        df_dedup.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(silver_table)
    
    else:
        
        df_dedup.createOrReplaceTempView("updates")
        
        spark.sql(f"""
        MERGE INTO {silver_table} t
        USING updates s
        ON t.{primary_key} = s.{primary_key}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """)
    
    print(f"✅ Processed {silver_table}")

In [0]:
process_silver_basic("customers", "customer_id")
process_silver_basic("products", "product_id")
process_silver_basic("orders", "order_id")
process_silver_basic("order_items", "order_item_id")